In [15]:
# Load processed data and recreate dependencies if running independently
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Load the processed dataframe
df_clean = pd.read_csv("../dataset/processed/processed_travel_data.csv")
df = df_clean  # Alias for compatibility

# Recreate label encoder
le_dest = LabelEncoder()
df_clean['Target_Destination'] = le_dest.fit_transform(df_clean['DestinationName'])

# Define features
numeric_features = ['Age', 'NumberOfAdults', 'NumberOfChildren', 'TravelMonth', 'Avg_Dest_Rating', 'Review_Count', 'Preference_Match_Score', 'Family_Size', 'Budget_Score']
categorical_features = ['Gender', 'Budget', 'Pref_Relaxation', 'Pref_Adventure', 'Pref_Culture', 'Pref_Spiritual', 'TravelSeason', 'Age_Group', 'Has_Children', 'Seasonal_Match']

X = df_clean[numeric_features + categorical_features]
y = df_clean['Target_Destination']

# Recreate preprocessor
preprocessor_enhanced = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features)
])

# Load the optimized models and results (assuming they are saved from hyperparameter_tuning.ipynb)
try:
    lr_opt = joblib.load("../models/optimized_logistic_regression_travel.pkl")
    rf_opt = joblib.load("../models/optimized_random_forest_travel.pkl")
    results_dict = joblib.load("../models/results_dict.joblib")
    print("Optimized models and results loaded successfully.")
except FileNotFoundError:
    print("Warning: Optimized models or results not found. Please run hyperparameter_tuning.ipynb first to save them.")
    lr_opt = None
    rf_opt = None
    results_dict = {}

print("Processed data and dependencies loaded.")

Optimized models and results loaded successfully.
Processed data and dependencies loaded.


# Demo

**Note**: This notebook assumes the optimized models from `hyperparameter_tuning.ipynb` are available. If running independently, load the saved models first.

---
## 7. Final Export

In [16]:
import os
import json

# Create models directory if it doesn't exist
os.makedirs("../models", exist_ok=True)

# Save optimized Random Forest model
rf_model_path = "../models/optimized_random_forest_travel.pkl"
joblib.dump(rf_opt, rf_model_path)

# Save label encoder
le_dest_path = "../models/label_encoder.joblib"
joblib.dump(le_dest, le_dest_path)

# Save preprocessor
preprocessor_path = "../models/preprocessor.joblib"
joblib.dump(preprocessor_enhanced, preprocessor_path)

# Prepare feature info for both models
categorical_values = {
    col: sorted(X[col].dropna().unique().tolist())
    for col in categorical_features
}

feature_info = {
    "all_features": list(X.columns),
    "categorical_features": categorical_features,
    "numeric_features": numeric_features,
    "categorical_values": categorical_values,
    "target_classes": le_dest.classes_.tolist()
}

# Save feature info
feature_info_path = "../models/feature_info_travel.json"
with open(feature_info_path, "w") as f:
    json.dump(feature_info, f, indent=4)

print("Optimized models, label encoder, preprocessor, and feature info saved successfully!")

print('\n✅ Exported artifacts:')
print('   - optimized_logistic_regression_travel.pkl   (Voting Classifier)')
print('   - optimized_random_forest_travel.pkl         (Random Forest)')
print('   - label_encoder.joblib                        (LabelEncoder)')
print('   - preprocessor.joblib                         (ColumnTransformer)')
print('   - feature_info_travel.json                    (Feature metadata)')

Optimized models, label encoder, preprocessor, and feature info saved successfully!

✅ Exported artifacts:
   - optimized_logistic_regression_travel.pkl   (Voting Classifier)
   - optimized_random_forest_travel.pkl         (Random Forest)
   - label_encoder.joblib                        (LabelEncoder)
   - preprocessor.joblib                         (ColumnTransformer)
   - feature_info_travel.json                    (Feature metadata)


In [17]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Interactive Demo Widgets
age = widgets.IntSlider(value=30, min=18, max=75, step=1, description='Age:')
gender = widgets.Dropdown(options=['Male', 'Female'], value='Male', description='Gender:')
budget = widgets.Dropdown(options=['Low', 'Medium', 'High'], value='Medium', description='Budget:')
num_adults = widgets.IntSlider(value=2, min=1, max=10, step=1, description='Adults:')
num_children = widgets.IntSlider(value=0, min=0, max=10, step=1, description='Children:')
travel_month = widgets.IntSlider(value=6, min=1, max=12, step=1, description='Travel Month:')
pref_relaxation = widgets.Checkbox(value=False, description='Relaxation')
pref_adventure = widgets.Checkbox(value=False, description='Adventure')
pref_culture = widgets.Checkbox(value=False, description='Culture')
pref_spiritual = widgets.Checkbox(value=False, description='Spiritual')

button = widgets.Button(description='Predict Destinations')
output = widgets.Output()

def predict_destinations(b):
    button.disabled = True  # Disable button during prediction
    with output:
        clear_output(wait=True)  # Use wait=True for better clearing
        
        # Create input data
        input_data = {
            'Age': age.value,
            'Gender': gender.value,
            'Budget': budget.value,
            'NumberOfAdults': num_adults.value,
            'NumberOfChildren': num_children.value,
            'TravelMonth': travel_month.value,
            'Pref_Relaxation': int(pref_relaxation.value),
            'Pref_Adventure': int(pref_adventure.value),
            'Pref_Culture': int(pref_culture.value),
            'Pref_Spiritual': int(pref_spiritual.value),
        }
        
        input_df = pd.DataFrame([input_data])
        
        # Compute derived features
        month = input_df['TravelMonth'].iloc[0]
        if 3 <= month <= 5:
            season = 'Spring'
        elif 6 <= month <= 8:
            season = 'Summer'
        elif 9 <= month <= 11:
            season = 'Fall'
        else:
            season = 'Winter'
        input_df['TravelSeason'] = season
        
        age_val = input_df['Age'].iloc[0]
        if age_val <= 25:
            ag = 'Young'
        elif age_val <= 35:
            ag = 'Adult'
        elif age_val <= 45:
            ag = 'Middle'
        elif age_val <= 55:
            ag = 'Senior'
        else:
            ag = 'Elder'
        input_df['Age_Group'] = ag
        
        input_df['Family_Size'] = input_df['NumberOfAdults'] + input_df['NumberOfChildren']
        input_df['Has_Children'] = (input_df['NumberOfChildren'] > 0).astype(int)
        
        budget_map = {'Low': 1, 'Medium': 2, 'High': 3}
        input_df['Budget_Score'] = input_df['Budget'].map(budget_map)
        
        input_df['Preference_Match_Score'] = 0  # Simplified
        input_df['Avg_Dest_Rating'] = df['Avg_Dest_Rating'].mean()
        input_df['Review_Count'] = df['Review_Count'].mean()
        input_df['Seasonal_Match'] = 0  # Simplified
        
        # Select features
        input_features = input_df[numeric_features + categorical_features]
        
        # Predict
        probs = rf_opt.predict_proba(input_features)
        top3_indices = np.argsort(probs[0])[-3:][::-1]
        top3_destinations = le_dest.inverse_transform(top3_indices)
        top3_probs = probs[0][top3_indices]
        
        print("🌍 Top 3 Recommended Destinations:")
        for i, (dest, prob) in enumerate(zip(top3_destinations, top3_probs)):
            print(f"{i+1}. {dest}: {prob:.3f}")
    

button.on_click(predict_destinations)

print("Travel Destination Recommendation Demo")
print("Adjust the inputs below and click 'Predict Destinations' to get recommendations using the optimized Random Forest model.")
display(age, gender, budget, num_adults, num_children, travel_month, pref_relaxation, pref_adventure, pref_culture, pref_spiritual, button, output)

Travel Destination Recommendation Demo
Adjust the inputs below and click 'Predict Destinations' to get recommendations using the optimized Random Forest model.


IntSlider(value=30, description='Age:', max=75, min=18)

Dropdown(description='Gender:', options=('Male', 'Female'), value='Male')

Dropdown(description='Budget:', index=1, options=('Low', 'Medium', 'High'), value='Medium')

IntSlider(value=2, description='Adults:', max=10, min=1)

...

Checkbox(value=False, description='Adventure')

Checkbox(value=False, description='Culture')

Checkbox(value=False, description='Spiritual')

Button(description='Predict Destinations', style=ButtonStyle())

Output()